# Havnsø – Probabilistic Pressure and Injectivity Screening

This is the second decision gate after the Havnsø static-capacity notebook. In every Monte Carlo iteration it checks:

1. Is sampled static capacity at least the project target?
2. Can each well support the selected injection rate?
3. Does the screening pressure remain at or below the pressure endpoint?

An iteration succeeds only when **all three criteria pass together**.

In [ ]:
#@title Install dependencies { display-mode: "form" }
%pip install -q "git+https://github.com/AnaSoles/ggg-co2-storage-eval.git" matplotlib pandas

In [ ]:
#@title Import libraries { display-mode: "form" }
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from storageeval import (
    Distribution, StorageSite, TechnicalScreeningCase,
    simulate, simulate_technical_screening,
)
plt.style.use("seaborn-v0_8-whitegrid")

## What is reported and what is derived

The pressure/injectivity calculation is a **GEUS-normalized screening surrogate**, not an Eclipse 100 reproduction. GEUS Report 2020/48 publishes a reference case and qualitative sensitivities, but no transferable pressure equation, numerical outcome for every sensitivity, or probability weights.

The surrogate exactly reaches the reference endpoint when all inputs equal the 2020 base case. It scales pressure demand with cumulative mass, total field rate, inverse permeability factor, and inverse N/G. It scales the per-well injectivity limit with permeability factor and N/G.

The updated 2023 static-capacity samples remain the current capacity evidence. The older version-0 dynamic result is used only to normalize this provisional technical gate.

## Editable inputs

The default **60 Mt** target is selected near the updated GEUS 2023/38 P50 of 62.82 Mt. It is an editable screening target, not a GEUS development plan.

In [ ]:
# Updated static-capacity inputs: GEUS 2023/38, Scenario 1.
grv_km3 = (2.9, 5.0, 8.0)
net_to_gross = (0.60, 0.75, 0.90)
porosity = (0.175, 0.219, 0.263)
co2_density_kg_m3 = (663.86, 698.8, 768.68)
storage_efficiency = (0.05, 0.10, 0.20)

# User-editable project controls.
target_mass_mt = 60.0 #@param {type:"number"}
number_of_wells = 3 #@param {type:"integer"}
rate_mtpy_per_well = 1.0 #@param {type:"number"}
iterations = 100000 #@param {type:"integer"}
random_seed = 42 #@param {type:"integer"}

# GEUS 2020/48 reference case and sensitivities.
initial_pressure_bar = 130.0
pressure_endpoint_bar = 240.0
reference_mass_mt = 270.0
reference_wells = 3
reference_rate_mtpy_per_well = 1.0
reference_net_to_gross = 0.5
permeability_factor = (0.5, 1.0, 2.0)

**Pressure interpretation:** the English summary calls 240 bar an “overpressure”. This notebook treats 240 bar as the absolute endpoint pressure. Adding 240 bar to the 130-bar initial pressure would conflict with the report's statement that the 75%-of-lithostatic fracture constraint was respected. Keep this interpretation visible until the original simulation files or an updated model resolve the terminology.

## Formulas, parameters, worked calculations and sources

### 1. Static capacity (storage capacity)

$$SC_i=GRV_i(N/G)_i\phi_i\rho_{CO_2,i}S_{eff,i}$$

| Symbol | Parameter / meaning | Unit | Distribution or default | Source / status |
|---|---|---:|---|---|
| $i$ | Monte Carlo iteration | – | $1,\ldots,N$ | Project method |
| $SC_i$ | Static CO₂ storage capacity in iteration $i$ | Mt CO₂ | Calculated | Equation used by GEUS static assessment |
| $GRV_i$ | Gross rock volume | km³ | PERT(2.9, 5.0, 8.0) | GEUS 2023/38, Table 8.2.3 |
| $(N/G)_i$ | Net-to-gross ratio | fraction | PERT(0.60, 0.75, 0.90) | GEUS 2023/38, Table 8.2.3 |
| $\phi_i$ | Porosity | fraction | PERT(0.175, 0.219, 0.263) | GEUS 2023/38, Table 8.2.3; analogue-based prognosis |
| $\rho_{CO_2,i}$ | In-situ CO₂ density | kg/m³ | PERT(663.86, 698.8, 768.68) | GEUS 2023/38, Table 8.2.3 |
| $S_{eff,i}$ | Storage-efficiency factor | fraction | PERT(0.05, 0.10, 0.20) | GEUS 2023/38, Table 8.2.3; screening assumption |

**Worked calculation using all modal values**

Because $1\ \mathrm{km^3}=10^9\ \mathrm{m^3}$ and $1\ \mathrm{Mt}=10^9\ \mathrm{kg}$, the conversion factors cancel numerically:

$$SC_{mode}=5.0\times0.75\times0.219\times698.8\times0.10=57.39\ \mathrm{Mt\ CO_2}$$

This single modal calculation is illustrative. The Monte Carlo result is obtained by sampling every input together in each iteration; it is not calculated by multiplying the final P50 values.

**Theory and source:** the volumetric capacity equation and Scenario 1 inputs follow GEUS 2023/38. This is the first decision gate: $SC_i\ge M_{target}$.

---

### 2. Injection duration

$$t=\frac{M_{target}}{nq_{well}}$$

| Symbol | Parameter / meaning | Unit | Notebook default | Source / status |
|---|---|---:|---:|---|
| $t$ | Time required to inject the target mass | years | Calculated | Mass ÷ total field rate |
| $M_{target}$ | User-selected storage target | Mt CO₂ | 60 | Project control; not a GEUS development target |
| $n$ | Number of injection wells | wells | 3 | Project control; equals GEUS reference count by default |
| $q_{well}$ | Selected rate per well | Mt CO₂/year/well | 1.0 | Project control; equals GEUS reference rate by default |

**Worked calculation**

$$t=\frac{60}{3\times1.0}=20\ \mathrm{years}$$

This formula is mass balance. It calculates project duration only; it does not decide whether pressure or injectivity passes.

---

### What “surrogate” means here

A **surrogate model** is a fast, simplified approximation used in place of a computationally expensive dynamic reservoir simulation. GEUS ran a 3-D Eclipse model. We do not have that model or a published response equation for every scenario, so this notebook uses transparent scaling equations anchored to the GEUS reference case.

The surrogate:

- reproduces the chosen reference endpoint when all inputs equal the GEUS base case;
- follows the reported sensitivity directions: more mass/rate raises pressure demand, whereas higher permeability and N/G reduce it;
- permits Monte Carlo screening in Python;
- **does not reproduce the spatial CO₂ plume, multiphase flow, pressure propagation, faults, completions or full well physics**.

Therefore, the surrogate equations below are original equations in this project. GEUS 2020/48 provides the reference values and qualitative sensitivity evidence, not these exact equations.

---

### 3. Pressure screening surrogate

$$R_{P,i}=\frac{M_{target}}{M_{ref}}\frac{nq_{well}}{n_{ref}q_{ref}}\frac{1}{k_{factor,i}}\frac{(N/G)_{ref}}{(N/G)_i}$$

$$P_{final,i}=P_0+(P_{lim}-P_0)R_{P,i}$$

| Symbol | Parameter / meaning | Unit | Distribution or default | Source / status |
|---|---|---:|---|---|
| $R_{P,i}$ | Normalized pressure-demand ratio | – | Calculated | Project surrogate |
| $M_{target}$ | Project target mass | Mt CO₂ | 60 | User-selected |
| $M_{ref}$ | Injected mass in reference case | Mt CO₂ | 270 | GEUS 2020/48 reference simulation |
| $n$, $q_{well}$ | Project wells and rate per well | wells; Mt/year/well | 3; 1.0 | User-selected |
| $n_{ref}$, $q_{ref}$ | Reference wells and rate per well | wells; Mt/year/well | 3; 1.0 | GEUS 2020/48 |
| $k_{factor,i}$ | Permeability multiplier relative to base case | – | PERT(0.5, 1.0, 2.0) | GEUS sensitivity multipliers converted by this project into an assumed probability distribution |
| $(N/G)_i$ | Sampled net-to-gross | fraction | PERT(0.60, 0.75, 0.90) | GEUS 2023/38 |
| $(N/G)_{ref}$ | Reference net-to-gross | fraction | 0.5 | GEUS 2020/48 base case |
| $P_0$ | Initial reservoir pressure | bar absolute | 130 | GEUS 2020/48 |
| $P_{lim}$ | Interpreted endpoint/pass limit | bar absolute | 240 | Project interpretation of GEUS wording |
| $P_{final,i}$ | Estimated final pressure in iteration $i$ | bar absolute | Calculated | Project surrogate output |

**Worked calculation using default controls and modal $k_{factor}=1$, $(N/G)=0.75$**

$$R_P=\frac{60}{270}\times\frac{3\times1}{3\times1}\times\frac{1}{1}\times\frac{0.5}{0.75}=0.1481$$

$$P_{final}=130+(240-130)\times0.1481=146.3\ \mathrm{bar\ absolute}$$

Since $146.3\le240$, this example passes the pressure gate.

**Where it comes from:** this exact formula is **not** in GEUS. It is a normalized project approximation. Its proportional directions are motivated by the GEUS permeability and N/G sensitivity runs and by Darcy-type flow behaviour, but only a calibrated dynamic model can establish the correct nonlinear response.

---

### 4. Per-well injectivity screening

$$q_{max,i}=q_{ref}k_{factor,i}\frac{(N/G)_i}{(N/G)_{ref}}$$

| Symbol | Parameter / meaning | Unit | Distribution or default | Source / status |
|---|---|---:|---|---|
| $q_{max,i}$ | Screening estimate of maximum supported rate per well | Mt CO₂/year/well | Calculated | Project surrogate |
| $q_{ref}$ | Reference rate per well | Mt CO₂/year/well | 1.0 | GEUS 2020/48 |
| $k_{factor,i}$ | Relative permeability multiplier | – | PERT(0.5, 1.0, 2.0) | GEUS sensitivities probabilized by this project |
| $(N/G)_i$ | Sampled net-to-gross | fraction | PERT(0.60, 0.75, 0.90) | GEUS 2023/38 |
| $(N/G)_{ref}$ | Reference net-to-gross | fraction | 0.5 | GEUS 2020/48 |
| $q_{well}$ | Requested project rate per well | Mt CO₂/year/well | 1.0 | User-selected pass threshold |

**Worked calculation using modal values**

$$q_{max}=1.0\times1.0\times\frac{0.75}{0.5}=1.5\ \mathrm{Mt/year/well}$$

The criterion is $q_{well}\le q_{max,i}$. Here, $1.0\le1.5$, so the example passes.

**Interpretation:** this estimates whether the requested per-well rate is below a normalized screening limit. It is not a well test, a productivity-index calculation, or a replacement for a radial/multiphase well model. The linear scaling is an explicit project assumption.

---

### 5. Pass criteria and probability

$$I_{success,i}=I(SC_i\ge M_{target})\land I(q_{well}\le q_{max,i})\land I(P_{final,i}\le P_{lim})$$

$$P(success)=\frac{1}{N}\sum_{i=1}^{N}I_{success,i}$$

| Symbol | Parameter / meaning | Unit |
|---|---|---:|
| $I(\cdot)$ | Indicator: 1 if a condition is true, otherwise 0 | binary |
| $I_{success,i}$ | 1 only when capacity **and** injectivity **and** pressure pass in the same iteration | binary |
| $\land$ | Logical AND; all connected conditions must be true | – |
| $N$ | Total Monte Carlo iterations | count |
| $P(success)$ | Successful iterations divided by $N$ | probability or % |

For example, if 49,963 of 100,000 iterations pass all three gates:

$$P(success)=\frac{49{,}963}{100{,}000}=0.49963=49.963\%$$

The three separate percentages are not averaged or blindly multiplied. The code preserves the shared $N/G$ sample and tests all gates in the same iteration.

## Stenlille porosity-permeability relationships

GEUS 2020/48 Figure 1 used conventional core-analysis data from Stenlille wells to populate Gassum Formation sandstone permeability in the version-0 dynamic model. With porosity expressed in percent:

$$k_{gas,upper}=0.000031\phi_{\%}^{4.91811}$$

$$k_{gas,lower}=0.00028\phi_{\%}^{4.91811}$$

GEUS then applied $k_{fluid}=0.5k_{gas}$. Permeability is in mD. These relationships are **Stenlille analogues, not Havnsø core measurements**. The chart below evaluates them across the updated 2023 Havnsø porosity range only to make their magnitude visible. The current surrogate still uses the dimensionless $k_{factor}$ distribution rather than these absolute permeability values. In this context, **mode** means the modal porosity input (21.9%), not a third porosity-permeability equation. The plot therefore highlights the minimum, mode and maximum porosity intersections on both GEUS curves rather than inventing an unsupported “mode permeability curve”.

In [ ]:
#@title Show Stenlille porosity-permeability analogue plot { display-mode: "form" }
def stenlille_permeability(phi_fraction, coefficient):
    phi_percent = np.asarray(phi_fraction) * 100.0
    gas_permeability_md = coefficient * phi_percent ** 4.91811
    return 0.5 * gas_permeability_md

case_names = ["Minimum", "Mode", "Maximum"]
case_colors = {"Minimum": "#1f77b4", "Mode": "#d62728", "Maximum": "#2ca02c"}
upper_case_k = stenlille_permeability(porosity, 0.000031)
lower_case_k = stenlille_permeability(porosity, 0.00028)

poro_perm_table = pd.DataFrame({
    "Porosity case": case_names,
    "Porosity (%)": np.array(porosity) * 100.0,
    "Upper Sands fluid k (mD)": upper_case_k,
    "Lower Sands fluid k (mD)": lower_case_k,
})
display(poro_perm_table.round(2))

phi_plot = np.linspace(porosity[0], porosity[2], 300)
fig, ax = plt.subplots(figsize=(12, 7))
ax.semilogy(
    phi_plot * 100,
    stenlille_permeability(phi_plot, 0.000031),
    label="Upper Sands relationship (GEUS)",
    linewidth=2.5,
)
ax.semilogy(
    phi_plot * 100,
    stenlille_permeability(phi_plot, 0.00028),
    label="Lower Sands relationship (GEUS)",
    linewidth=2.5,
)

for case_name, phi, upper_k, lower_k in zip(case_names, porosity, upper_case_k, lower_case_k):
    x = phi * 100
    color = case_colors[case_name]
    line_width = 2.4 if case_name == "Mode" else 1.4
    ax.axvline(
        x, color=color, linestyle="--", linewidth=line_width, alpha=0.85,
        label=f"{case_name} porosity = {x:.1f}%" if case_name == "Mode" else None,
    )
    ax.scatter([x, x], [upper_k, lower_k], color=color, s=70, zorder=5)
    ax.annotate(
        f"{case_name}: {x:.1f}%, {upper_k:.1f} mD",
        (x, upper_k), xytext=(7, -15), textcoords="offset points",
        fontsize=9, color=color,
    )
    ax.annotate(
        f"{case_name}: {x:.1f}%, {lower_k:.1f} mD",
        (x, lower_k), xytext=(7, 7), textcoords="offset points",
        fontsize=9, color=color,
    )

ax.set(
    xlabel="Porosity (%)",
    ylabel="Estimated fluid permeability (mD, logarithmic scale)",
    title="Stenlille Gassum Formation analogues: porosity–permeability intersections",
)
ax.legend(loc="upper left")
fig.tight_layout()
plt.show()

In [ ]:
#@title Run static capacity Monte Carlo { display-mode: "form" }
site = StorageSite(
    name="Havnsø – Gassum Formation – Scenario 1",
    grv=Distribution.pert(*grv_km3),
    net_to_gross=Distribution.pert(*net_to_gross),
    porosity=Distribution.pert(*porosity),
    co2_density=Distribution.pert(*co2_density_kg_m3),
    storage_efficiency=Distribution.pert(*storage_efficiency),
)
capacity_result = simulate(site, iterations=iterations, seed=random_seed)

## Run pressure and injectivity screening

This is **not another formula and not an additional parameter table**. It is the execution step that applies the pressure, injectivity and pass/fail formulas above to every Monte Carlo iteration.

The code:

1. packages the project controls and GEUS reference values into `TechnicalScreeningCase`;
2. draws one $k_{factor,i}$ value from PERT(0.5, 1, 2) for each iteration;
3. reuses the same $(N/G)_i$ and $SC_i$ values already generated by the static-capacity simulation;
4. calculates $R_{P,i}$, $P_{final,i}$ and $q_{max,i}$;
5. creates three Boolean arrays—capacity pass, injectivity pass and pressure pass;
6. combines them with logical AND to produce $I_{success,i}$.

The returned `technical_result` stores all iteration-level values. The next cell summarizes those arrays; it does not introduce new scientific assumptions.

In [ ]:
#@title Run pressure and injectivity screening { display-mode: "form" }
technical_case = TechnicalScreeningCase(
    name=f"Havnsø – {target_mass_mt:g} Mt technical screening",
    target_mass_mt=target_mass_mt,
    wells=number_of_wells,
    rate_mtpy_per_well=rate_mtpy_per_well,
    permeability_factor=Distribution.pert(*permeability_factor),
    initial_pressure_bar=initial_pressure_bar,
    pressure_limit_bar=pressure_endpoint_bar,
    reference_mass_mt=reference_mass_mt,
    reference_wells=reference_wells,
    reference_rate_mtpy_per_well=reference_rate_mtpy_per_well,
    reference_net_to_gross=reference_net_to_gross,
)
technical_result = simulate_technical_screening(
    technical_case, capacity_result, seed=random_seed + 1
)

## Integrated technical-screening result

This table is a summary of the Monte Carlo arrays created above. For every pass metric, the notebook now shows the numerator, denominator and equation used.

Pressure percentiles use ordinary statistical percentiles: “P90 final pressure” is the 90th percentile, meaning 90% of simulated pressures are at or below that value. This differs from the common capacity convention in which P90 denotes a conservative low-capacity estimate.

In [ ]:
#@title Show integrated technical-screening result and calculations { display-mode: "form" }
summary = technical_result.summary()
n_iterations = technical_result.success.size
capacity_pass_count = int(technical_result.capacity_pass.sum())
injectivity_pass_count = int(technical_result.injectivity_pass.sum())
pressure_pass_count = int(technical_result.pressure_pass.sum())
success_count = int(technical_result.success.sum())
failure_count = n_iterations - success_count

result_table = pd.DataFrame([
    ["Project target", "User input", f"{target_mass_mt:.1f} Mt"],
    ["Injection duration", f"{target_mass_mt:g} / ({number_of_wells} × {rate_mtpy_per_well:g})", f"{technical_result.duration_years:.1f} years"],
    ["Capacity passes", f"{capacity_pass_count:,} / {n_iterations:,}", f"{summary['capacity_pass_probability']:.1%}"],
    ["Injectivity passes", f"{injectivity_pass_count:,} / {n_iterations:,}", f"{summary['injectivity_pass_probability']:.1%}"],
    ["Pressure passes", f"{pressure_pass_count:,} / {n_iterations:,}", f"{summary['pressure_pass_probability']:.1%}"],
    ["All criteria pass (technical success)", f"{success_count:,} / {n_iterations:,}; capacity AND injectivity AND pressure", f"{summary['success_probability']:.1%}"],
    ["At least one criterion fails", f"{failure_count:,} / {n_iterations:,}; 1 − P(success)", f"{summary['failure_probability']:.1%}"],
    ["P90 final pressure", "90th percentile of all P_final values", f"{summary['p90_final_pressure_bar']:.1f} bar"],
    ["P50 final pressure", "Median (50th percentile) of all P_final values", f"{summary['p50_final_pressure_bar']:.1f} bar"],
], columns=["Metric", "How it is calculated", "Result"])
result_table

In [ ]:
#@title Show criterion pass probabilities { display-mode: "form" }
fig, ax = technical_result.plot_criteria()
plt.show()

In [ ]:
#@title Show capacity and injectivity decision limits { display-mode: "form" }
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(capacity_result.capacity_mt, bins=50, color="#9ecae1", edgecolor="white")
axes[0].axvline(target_mass_mt, color="#c00000", linestyle="--", linewidth=2, label=f"Target: {target_mass_mt:g} Mt")
axes[0].set(xlabel="Static capacity (Mt)", ylabel="Iterations", title="Capacity gate")
axes[0].legend()

axes[1].hist(technical_result.injectivity_limit_mtpy_per_well, bins=50, color="#a1d99b", edgecolor="white")
axes[1].axvline(rate_mtpy_per_well, color="#c00000", linestyle="--", linewidth=2, label=f"Selected rate: {rate_mtpy_per_well:g} Mt/year/well")
axes[1].set(xlabel="Screening injectivity limit (Mt/year/well)", ylabel="Iterations", title="Injectivity gate")
axes[1].legend()
fig.tight_layout()
plt.show()

In [ ]:
#@title Show final-pressure distribution and limit { display-mode: "form" }
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(technical_result.final_pressure_bar, bins=50, color="#9ecae1", edgecolor="white")
ax.axvline(pressure_endpoint_bar, color="#c00000", linestyle="--", linewidth=2, label=f"Pressure endpoint: {pressure_endpoint_bar:g} bar")
ax.set(xlabel="Screening final pressure (bar absolute)", ylabel="Iterations", title=technical_case.name)
ax.legend()
plt.show()

## Bibliographic references

### Formula provenance at a glance

| Notebook component | Provenance |
|---|---|
| Static capacity equation and Scenario 1 ranges | GEUS 2023/38 |
| Stenlille porosity–permeability equations | GEUS 2020/48, Figure 1; underlying core database in GEUS 2020/28 |
| Injection duration | Direct mass balance: mass divided by total field rate |
| Pressure surrogate | Original GPPEVAL-CO₂ scaling equation, normalized to GEUS 2020/48 |
| Per-well injectivity surrogate | Original GPPEVAL-CO₂ scaling equation, normalized to GEUS 2020/48 |
| Pass probabilities | Empirical Monte Carlo frequency calculated by this notebook |

- Nielsen, C.M. (2020). *Dynamic storage capacity evaluation for the Hanstholm and Havnsø structures*. GEUS Report 2020/48. [Official PDF](https://data.geus.dk/pure-pdf/GEUS-R_2020_48_web.pdf). Source for the Eclipse workflow, Figure 1 porosity-permeability equations, reference injection case, pressure constraint and sensitivity scenarios.
- Kristensen, L. (2020). *Reservoir data - Stenlille area*. GEUS Report 2020/28. [Official PDF](https://data.geus.dk/pure-pdf/GEUS-R_2020_28_web.pdf). Primary report for the Stenlille log/core database and porosity-permeability analogues extrapolated to Havnsø.
- Gregersen, U., Vosgerau, H., Smit, F.W.H., et al. (2023). *The Havnsø structure - Seismic data and interpretation to mature potential geological storage of CO₂*. GEUS Report 2023/38. [DOI](https://doi.org/10.22008/gpub/34705). Source for the updated Scenario 1 static-capacity inputs.

GEUS 2020/48 states that its simulator solves governing flow equations based on Darcy's equation. This notebook does **not** solve that dynamic Darcy-flow system. The normalized pressure and injectivity surrogate equations are original equations in this project; GEUS supplies their reference point and qualitative sensitivity directions, not the equations themselves.

## Interpretation and next calibration step

The combined success percentage is a **probabilistic screening result**, not yet a calibrated geological-risk probability. Capacity uncertainty comes from GEUS 2023/38. The permeability-factor probability distribution is derived from the 0.5× and 2× sensitivity scenarios in GEUS 2020/48; GEUS did not assign probabilities to them.

Before using the result for a project decision, replace this surrogate with an updated dynamic reservoir model based on the 2023 geometry and obtain numerical pressure responses, site-specific permeability/relative-permeability data, fracture-pressure measurements, and well-design constraints.